In [1]:
import torch
import torch.nn as nn
from d2l import torch as d2l

In [2]:
import math
import os
import random
import collections
from ptbData import load_data_ptb

In [ ]:
batch_size=512
data_iter,vocab = load_data_ptb(batch_size,5,5)

In [ ]:


def skip_gram(center,context_negative,embed_v,embed_u):
    v=embed_v(center)
    u=embed_u(context_negative)

    return torch.bmm(v,u.permute(0,2,1)).squeeze(dim=1)

def sigmoidBSE(pred,mask,label):
    return nn.functional.binary_cross_entropy_with_logits(pred,target=label,weight=mask,reduction='none')/mask.sum(dim=1)

embed_size=100
embeds=nn.Sequential(nn.Embedding(len(vocab),embed_size),nn.Embedding(len(vocab),embed_size))

def train(net,lr,data_iter,epochs,device):
    def init_weight(m):
        if type(m) == nn.Embedding:
            nn.init.xavier_uniform_(m.weight)

    net.apply(init_weight)
    net=net.to(device)
    updater=torch.optim.Adam(net.parameters(),lr=lr)
    for epoch in range(0,epochs):
        for center,context_negative,mask,label in data_iter:
            updater.zero_grad()
            l=sigmoidBSE(skip_gram(center.to(device),context_negative.to(device),net[0],net[1]),mask.to(device),label.to(device))
            l.sum().backward()
            with torch.no_grad():
                updater.step()
        print(f"epoch {epoch} loss:{l.mean()}")

In [23]:
train(embeds,0.002,data_iter,5,torch.device('cuda:2'))

epoch 1 loss:49.174983978271484
epoch 2 loss:46.40757751464844
epoch 3 loss:41.82668685913086
epoch 4 loss:43.0594367980957
